# Developing helper functions for the pipeline

In [1]:
import pandas as pd

df = pd.read_csv('../../data/raw/hyperbook.csv')

In [3]:
title, content = df[['title', 'content']]

In [3]:
from keybert import KeyBERT
from langchain_text_splitters import RecursiveCharacterTextSplitter
from collections import Counter

model_name = 'Qwen/Qwen3-Embedding-0.6B'
keybert = KeyBERT(model=model_name)
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(model_name='gpt-4', chunk_size=1024, chunk_overlap=128)

def get_keywords(doc: str) -> list[str]:
    """
    Splits and embeds doc and returns the top 10
    most semantically relevant keywords
    """

    chunks = splitter.split_text(doc)
    result = keybert.extract_keywords(chunks)
    all_keywords = [kw[0] for chunk_keywords in result for kw in chunk_keywords]
    top_keywords = Counter(all_keywords).most_common(10)
    
    return top_keywords

In [13]:
from sentence_transformers import SentenceTransformer
from collections import defaultdict
from hdbscan import HDBSCAN

model_name = 'Qwen/Qwen3-Embedding-0.6B'
model = SentenceTransformer(model_name)
clusterer = HDBSCAN(min_cluster_size=2, metric='euclidean')

def cluster_keywords(keywords: list[str]) -> dict:
    """
    Clusters keywords into semantic clusters
    and returns a map of their members
    """
    embeddings = model.encode(keywords)
    clusters = clusterer.fit_predict(embeddings)
    cluster_map = defaultdict(list)
    embeddings_map = {}

    for keyword, cluster_id in zip(keywords, clusters):
        if cluster_id != -1: # Skip noise if using HDBSCAN
            cluster_map[int(cluster_id)].append(keyword)
            embeddings_map[keyword] = embeddings[keywords.index(keyword)]

    return cluster_map, embeddings_map

In [14]:
keywords = ['glycoprotein', 'protein', 'glycoproteins', 'mussels', 'clams', 'mollusks', 'robot', 'android', 'artificial intelligence']

cluster_keywords(keywords)

d:\Projects\hyperbook-weaver\venv\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
d:\Projects\hyperbook-weaver\venv\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


(defaultdict(list,
             {0: ['glycoprotein', 'protein', 'glycoproteins'],
              1: ['mussels', 'clams', 'mollusks']}),
 {'glycoprotein': array([-0.03381737, -0.02381012, -0.01595601, ...,  0.00041461,
         -0.01179637,  0.01176754], dtype=float32),
  'protein': array([-0.00459065, -0.04338182, -0.01411273, ..., -0.01708603,
         -0.00218913,  0.03300722], dtype=float32),
  'glycoproteins': array([-0.04787508, -0.03137838, -0.01602631, ..., -0.00337595,
         -0.01681524,  0.01844322], dtype=float32),
  'mussels': array([-0.06903044, -0.0236024 , -0.01246766, ...,  0.00068482,
         -0.00723394, -0.01174365], dtype=float32),
  'clams': array([-0.05949921, -0.00083166, -0.01345477, ..., -0.01180673,
         -0.00942874, -0.01803245], dtype=float32),
  'mollusks': array([-0.0519052 ,  0.00618933, -0.01111559, ...,  0.00023069,
         -0.00674414,  0.02556807], dtype=float32)})

In [ ]:
import pandas as pd

df = pd.read_csv('../../data/raw/hyperbook.csv', index_col='url')
sample = df.iloc[0]
content = sample['content']

In [2]:
import plotly.graph_objects as go
import numpy as np

# Sample data
x = np.random.rand(50)
y = np.random.rand(50)
z = np.random.rand(50)

# Create a 3D scatter trace
trace = go.Scatter3d(
    x=x, y=y, z=z,
    mode='markers',
    marker=dict(size=5, color=z, colorscale='Viridis', opacity=0.8),
)

# Create a figure and add the trace
fig = go.Figure(data=[trace])

# Customize layout
fig.update_layout(
    title='Simple 3D Scatter Plot',
    scene=dict(
        xaxis_title='X-axis',
        yaxis_title='Y-axis',
        zaxis_title='Z-axis'
    ),
    height=1000
)

# Display the plot
fig.show()

In [18]:
from sentence_transformers import SentenceTransformer, util

model_name = 'Qwen/Qwen3-Embedding-0.6B'

def name_cluster(keywords: list[str]) -> str:
    """
    Names a cluster of keywords by finding the keyword
    closest to the centroid of the embeddings.
    """

    model = SentenceTransformer(model_name)
    embeddings = model.encode(keywords, convert_to_tensor=True, normalize_embeddings=True)

    # Calculate the arithmetic mean and mormalize
    centroid = util.normalize_embeddings(embeddings.mean(dim=0, keepdim=True))
    # Get the embedding closest to the centroid
    name_index = int(util.cos_sim(centroid, embeddings)[0].argmax())

    return keywords[name_index]

In [19]:
keywords = ['glycoprotein', 'protein', 'glycoproteins']

In [20]:
name_cluster(keywords)

'glycoproteins'